# Create Tables

In [0]:
CREATE OR REPLACE TABLE mcp.game.amg_updates (
   update_key STRING NOT NULL COMMENT 'Unique key for the update, usually the calendar year and month in YYYYMM format.'
  ,effective_date DATE COMMENT 'Approximate date when the update was released/effective.'
  ,update_notes STRING COMMENT 'Notes about the update.'
)
 COMMENT 'This table contains information about character and other card updates.';

ALTER TABLE mcp.game.amg_updates
ADD CONSTRAINT pk_game_amg_updates PRIMARY KEY (update_key);

--Add Known Updates
INSERT INTO mcp.game.amg_updates (update_key, effective_date)
VALUES
    ('202606','2026-06-30')
   ,('202511','2025-11-30')
   ,('202505','2025-05-31')
   ,('202502','2025-02-28')
   ,('202309','2023-09-30')
   ,('202111','2021-11-30')

In [0]:
CREATE OR REPLACE TABLE mcp.game.characters (
   character_id STRING NOT NULL COMMENT 'Unique identifier for the character. Should match the MTC code from Cerebro and Jarvis.'
  ,character_name STRING COMMENT 'Name of the character on the card.'
  ,alter_ego STRING COMMENT 'Name of the charcter\'s alter ego'
  ,threat INT COMMENT 'The *threat* cost of the character.'
  ,front_health INT COMMENT 'Health on the front (healthy) side of the character card.'
  ,back_health INT COMMENT 'Health on the back (injured) side of the character card.'
  ,extra_power INT COMMENT 'Additional power this charcter received in the power phase.'
  ,leadership_affiliation_id STRING COMMENT 'ID of the affilaition this charcter this character leads.'
  ,leadership_card_id STRING COMMENT 'ID for the team tactics cards required for this charcter to have a leadership.'
  ,gem_bearer BOOLEAN COMMENT 'Indicates if the character can be assigned an Infinity Gem.'
  ,gem_limit INT COMMENT 'The maximum number of infinity games that can be assigned to this character.'
  ,effective_date DATE COMMENT 'Approximate date when the character was released or the update was released.'
  ,update_key STRING COMMENT 'Unique key for the update, usually the calendar year and month in YYYYMM format. Will be *NULL* if the charcter has not been updated.'
  ,current_version BOOLEAN NOT NULL COMMENT 'Indicates the current version of the character and should be *TRUE* for the latest version.'
)
  COMMENT 'Contains information about the characters in Marvel Crisis Protocol.';

-- Set Primary Key
ALTER TABLE mcp.game.characters
ADD CONSTRAINT pk_game_characters PRIMARY KEY (character_id);

-- Enable default values for columns
ALTER TABLE mcp.game.characters SET TBLPROPERTIES (
  'delta.feature.allowColumnDefaults' = 'enabled'
);

-- Set default value for current_version
ALTER TABLE mcp.game.characters ALTER COLUMN current_version SET DEFAULT FALSE;

In [0]:
CREATE OR REPLACE TABLE mcp.game.infinity_gems (
  gem_id STRING NOT NULL COMMENT 'Unique identifier for the  Infinity Gem. Should match the MTC code from Cerebro and Jarvis.'
 ,gem_name STRING COMMENT 'Name of the  Infinity Gem.'
 ,alternate_gem_id INT COMMENT 'Alternate ID for the Infinity Gem.'
 ,color STRING COMMENT 'The color of the  Infinity Gem.'
 ,threat INT COMMENT 'The additional threat cost to take the Infinity Gem.'
)
COMMENT 'Contains information about the infinity gems in Marvel Crisis Protocol.';

ALTER TABLE mcp.game.infinity_gems
ADD CONSTRAINT pk_game_infinity_gems_gem_id PRIMARY KEY (gem_id);

In [0]:
CREATE OR REPLACE TABLE mcp.game.affiliations (
   affiliation_name STRING NOT NULL COMMENT 'Name of the affiliation.'
)
COMMENT 'Contains information about the affiliations in Marvel Crisis Protocol.';

ALTER TABLE mcp.game.affiliations
ADD CONSTRAINT pk_game_affilations_affilaition_name PRIMARY KEY (affiliation_name);

In [0]:
CREATE OR REPLACE TABLE mcp.game.affiliated_charcters (
   affiliation_name STRING NOT NULL COMMENT 'Name of the affiliation.'
  ,character_id STRING NOT NULL COMMENT 'Unique identifier for the character. Should match the MTC code from Cerebro and Jarvis.' 
)
  COMMENT 'Matches characters with their affiliations.';

ALTER TABLE mcp.game.affiliated_charcters
ADD CONSTRAINT pk_game_affiliated_charcters PRIMARY KEY (affiliation_name, character_id);

-- Process to Add Foreign Keys
--First, drop the foreign keys if they already exists
ALTER TABLE mcp.game.affiliated_charcters DROP FOREIGN KEY IF EXISTS (affiliation_name);
ALTER TABLE mcp.game.affiliated_charcters DROP FOREIGN KEY IF EXISTS (character_id);

--add foreign keys
ALTER TABLE mcp.game.affiliated_charcters
  ADD CONSTRAINT fk_mcp_game_affiliated_charcters_affiliation_name 
  FOREIGN KEY (affiliation_name) 
  REFERENCES mcp.game.affiliations(affiliation_name);

ALTER TABLE mcp.game.affiliated_charcters 
  ADD CONSTRAINT fk_mcp_game_affiliated_charcters_character_id 
  FOREIGN KEY (character_id)
  REFERENCES mcp.game.characters(character_id);

In [0]:
CREATE OR REPLACE TABLE mcp.game.charcters_gems (
   gem_id STRING NOT NULL COMMENT 'Unique identifier for the  Infinity Gem. Should match the MTC code from Cerebro and Jarvis.'
  ,character_id STRING NOT NULL COMMENT 'Unique identifier for the character. Should match the MTC code from Cerebro and Jarvis.' 
)
  COMMENT 'Matches characters with the gems they can hold.';

ALTER TABLE mcp.game.charcters_gems
ADD CONSTRAINT pk_game_charcters_gems PRIMARY KEY (gem_id, character_id);

-- Process to Add Foreign Keys
--First, drop the foreign keys if they already exists
ALTER TABLE mcp.game.charcters_gems DROP FOREIGN KEY IF EXISTS (gem_id);
ALTER TABLE mcp.game.charcters_gems DROP FOREIGN KEY IF EXISTS (character_id);

--add foreign keys
ALTER TABLE mcp.game.charcters_gems 
  ADD CONSTRAINT fk_mcp_game_characters_game_affiliation_name 
  FOREIGN KEY (gem_id) 
  REFERENCES mcp.game.infinity_gems(gem_id);

ALTER TABLE mcp.game.charcters_gems 
  ADD CONSTRAINT fk_mcp_game_charters_gems_character_id 
  FOREIGN KEY (character_id)
  REFERENCES mcp.game.characters(character_id);

In [0]:
CREATE OR REPLACE TABLE mcp.game.character_updates (
   update_key STRING NOT NULL COMMENT 'Unique key for the update, usually the calendar year and month in YYYYMM format.'
  ,character_id STRING NOT NULL COMMENT 'Unique identifier for the character. Should match the MTC code from Cerebro and Jarvis.'
)
  COMMENT 'Indicates if a charcter was part of a particular update.';

ALTER TABLE mcp.game.character_updates
ADD CONSTRAINT pk_game_character_updates PRIMARY KEY (update_key, character_id);

In [0]:
CREATE OR REPLACE TABLE mcp.game.product_codes (
   sku STRING NOT NULL COMMENT 'Unique identifier for the product, usually starts with CP or MCP.'
  ,release_date DATE COMMENT 'Approximate US release date for the product.'
)
  COMMENT 'Information about MCP products including their release date.';

ALTER TABLE mcp.game.product_codes
ADD CONSTRAINT pk_game_product_codes PRIMARY KEY (sku);

In [0]:
CREATE OR REPLACE TABLE mcp.game.product_to_character (
  sku STRING NOT NULL COMMENT 'Unique identifier for the product, usually starts with CP or MCP.'
 ,character_id STRING NOT NULL COMMENT 'Unique identifier for the character. Should match the MTC code from Cerebro and Jarvis.'
)
  COMMENT 'Matches products with characters.';

ALTER TABLE mcp.game.product_to_character
ADD CONSTRAINT pk_game_product_to_character PRIMARY KEY (sku, character_id);

In [0]:
--First, drop the foreign keys if they already exists
ALTER TABLE mcp.game.character_updates DROP FOREIGN KEY IF EXISTS (update_key);
ALTER TABLE mcp.game.character_updates DROP FOREIGN KEY IF EXISTS (character_id);

--add foreign keys
ALTER TABLE mcp.game.character_updates 
  ADD CONSTRAINT fk_mcp_game_character_updates_update_key 
  FOREIGN KEY (update_key) 
  REFERENCES mcp.game.amg_updates(update_key);

ALTER TABLE mcp.game.character_updates 
  ADD CONSTRAINT fk_mcp_game_character_updates_character_id 
  FOREIGN KEY (character_id)
  REFERENCES mcp.game.characters(character_id);

In [0]:
--First, drop the foreign keys if they already exists
ALTER TABLE mcp.game.product_to_character DROP FOREIGN KEY IF EXISTS (sku);
ALTER TABLE mcp.game.product_to_character DROP FOREIGN KEY IF EXISTS (character_id);

--Then, add the foreign keys
ALTER TABLE mcp.game.product_to_character 
  ADD CONSTRAINT fk_mcp_game_product_to_character_sku 
  FOREIGN KEY (sku) 
  REFERENCES mcp.game.product_codes(sku);

ALTER TABLE mcp.game.product_to_character 
  ADD CONSTRAINT fk_mcp_game_product_to_character_character_id 
  FOREIGN KEY (character_id)
  REFERENCES mcp.game.characters(character_id);